2.3.3 Tensor dot

In [ ]:
import numpy as np

In [ ]:
x= np.array([[1,2,3],[4,5,6]])
y=np.array([7,8,9])

In [ ]:
print(y.shape[0])

In [ ]:
def naive__matrix_vector_dot(x,y):
  assert len(x.shape)==2
  assert len(y.shape)==1
  assert x.shape[1]== y.shape[0]

  z=np.zeros(x.shape[0])
  for i in range (x.shape[0]):
    for j in range (x.shape[1]):
      z[i] +=x[i,j]*y[j]
  return z

In [ ]:
z = naive__matrix_vector_dot(x,y)
print(z)

In [ ]:
t = np.dot(x, y)
t

In [ ]:
q =np.array([ # 1st level Array (Outer)
    [ # 2nd level Array
        [[1, 2, 20], [3, 4,20]], # 3rd level arrays, containing 2 4th level arrays
        [[5, 6, 20], [7, 8, 20]]
    ],
    [ # 2nd Level array
        [[9, 10, 20], [11, 12, 20]],
        [[13, 14, 20], [15, 16, 20]]
    ]
])
q.shape

In [ ]:
p= np.array([2,3,4])
p.shape

In [ ]:
t = np.dot(q,p )
t.shape

In [ ]:
A= np.array([0.5,1])
B =np.array([1,0.25])
C= A + B
C

**3. Getting started with neural networks**

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "torch"

import keras

print("Keras version:", keras.__version__)
print("Keras backend:", keras.backend.backend())


**3.1 Loading the IMDB dataset**

In [ ]:
from keras.datasets import imdb

In [ ]:
(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=10000 )

The argument num_words=10000 means you’ll only keep the top 10,000 most frequently occurring words in the training data. Rare words will be discarded. This allows
you to work with vector data of manageable size.
 The variables train_data and test_data are lists of reviews; each review is a list of
word indices (encoding a sequence of words). train_labels and test_labels are
lists of 0s and 1s, where 0 stands for negative and 1 stands for positive

In [ ]:
train_data.shape

In [ ]:
ln_Lists= []
for line in test_data:
    ln_Lists.append( len(line))
max(ln_Lists)

In [ ]:
train_data[0]

In [ ]:
python_lists = [
    [1, 2, 3],
    [0],
    [4, 5],
    [6],
    [7, 8, 9, 10],
]

maximum_value = max(
    max(sequence)
    for sequence in python_lists
)

print("Maximum value:", maximum_value)

In [ ]:
 train_labels[0]

Because you’re restricting yourself to the top 10,000 most frequent words, no word index will exceed 10,000:

In [ ]:
max([max(sequence) for sequence in train_data])

In [ ]:
train_data.shape

For kicks, here’s how you can quickly decode one of these reviews back to English words

In [ ]:
word_index = imdb.get_word_index()
reverse_word_index = dict(
[(value, key) for (key, value) in word_index.items()])
decoded_review = ' '.join(
[reverse_word_index.get(i - 3, '?') for i in train_data[0]])

In [ ]:
decoded_review

**3.2 Encoding the integer sequences into a binary matrix**

In [ ]:
def vectorize_sequences(sequences, dimension=10000):
  results = np.zeros((len(sequences), dimension))
  for i, sequence in enumerate(sequences):
    results[i, sequence] = 1.
  return results

In [ ]:
x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)

In [ ]:
x_train.shape

Here’s what the samples look like now:

In [ ]:
 x_test.shape

You should also vectorize your labels, which is straightforward:

In [ ]:
y_train = np.asarray(train_labels).astype('float32')
y_test = np.asarray(test_labels).astype('float32')

**3.3 The model definition**

In [ ]:
from keras import models
from keras import layers

In [ ]:

model = models.Sequential()
model.add(layers.Dense(16, activation='relu', input_shape=(10000,)))
model.add(layers.Dense(16, activation='relu'))
model.add(layers.Dense(1, activation='sigmoid'))

In [ ]:
model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
from keras import losses
from keras import metrics
model.compile(optimizer=keras.optimizers.RMSprop(learning_rate=0.001), loss=losses.binary_crossentropy, metrics=[metrics.binary_accuracy])

**3.7 Setting aside a validation set**

In [ ]:
x_val = x_train[:10000]
partial_x_train = x_train[10000:]
y_val = y_train[:10000]
partial_y_train = y_train[10000:]

**3.8 Training your model**

In [ ]:
model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['acc'])
history = model.fit(partial_x_train, partial_y_train, epochs=20, batch_size=512, validation_data=(x_val, y_val))

In [ ]:
history_dict = history.history
history_dict.keys()


**3.9 Plotting the training and validation loss**

In [ ]:
import matplotlib.pyplot as plt

history_dict = history.history
loss_values = history_dict['loss']
val_loss_values = history_dict['val_loss']
epochs = range(1, len( history_dict['acc']) + 1)
plt.plot(epochs, loss_values, 'bo', label='Training loss')
plt.plot(epochs, val_loss_values, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

**3.10 Plotting the training and validation accuracy**

In [ ]:
plt.clf()
acc_values = history_dict['acc']
val_acc_values = history_dict['val_acc']
plt.plot(epochs, acc_values, 'bo', label='Training acc')
plt.plot(epochs, val_acc_values, 'b', label='Validation acc')
plt.title('Training and validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

**3.11 Retraining a model from scratch**

to prevent overfitting, you could stop training after three epochs. I

In [ ]:
model = models.Sequential()
model.add(layers.Dense(16, activation='relu', input_shape=(10000,)))
model.add(layers.Dense(16, activation='relu'))
model.add(layers.Dense(1, activation='sigmoid'))
model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=4, batch_size=512)
results = model.evaluate(x_test, y_test)

In [ ]:
results

**3.4.5 Using a trained network to generate predictions on new data**

In [ ]:
 model.predict(x_test)

**3.4.6 Further experiment**

- You used two hidden layers. Try using one or three hidden layers, and see how
doing so affects validation and test accuracy.


In [ ]:
model = models.Sequential()
model.add(layers.Dense(16, activation='relu', input_shape=(10000,)))
model.add(layers.Dense(16, activation='relu'))
model.add(layers.Dense(16, activation='relu'))
model.add(layers.Dense(1, activation='sigmoid'))
model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=4, batch_size=512)
results = model.evaluate(x_test, y_test)

In [ ]:
results

- Try using layers with more hidden units or fewer hidden units: 32 units, 64 units,and so on.

In [ ]:
model = models.Sequential()
model.add(layers.Dense(32, activation='relu', input_shape=(10000,)))
model.add(layers.Dense(32, activation='relu'))
model.add(layers.Dense(32, activation='relu'))
model.add(layers.Dense(1, activation='sigmoid'))
model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=4, batch_size=512)
results = model.evaluate(x_test, y_test)

In [ ]:
results

In [ ]:
model = models.Sequential()
model.add(layers.Dense(64, activation='relu', input_shape=(10000,)))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(1, activation='sigmoid'))
model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=4, batch_size=512)
results = model.evaluate(x_test, y_test)

In [ ]:
results

- Try using the mse loss function instead of binary_crossentropy.

In [ ]:
model = models.Sequential()
model.add(layers.Dense(64, activation='relu', input_shape=(10000,)))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(1, activation='sigmoid'))
model.compile(optimizer='rmsprop', loss='mse', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=4, batch_size=512)
results = model.evaluate(x_test, y_test)

In [ ]:
results

- Try using the tanh activation (an activation that was popular in the early days of neural networks) instead of relu

In [ ]:
model = models.Sequential()
model.add(layers.Dense(64, activation='tanh', input_shape=(10000,)))
model.add(layers.Dense(64, activation='tanh'))
model.add(layers.Dense(1, activation='sigmoid'))
model.compile(optimizer='rmsprop', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, epochs=4, batch_size=512)
results = model.evaluate(x_test, y_test)

In [ ]:
results

# **3.5 Classifying newswires: a multiclass classification example**

**3.12 Loading the Reuters dataset**

In [ ]:
from keras.datasets import reuters
(train_data, train_labels), (test_data, test_labels) = reuters.load_data(num_words=10000)

As with the IMDB dataset, the argument num_words=10000 restricts the data to the 10,000 most frequently occurring words found in the data.  You have 8,982 training examples and 2,246 test examples:

In [ ]:
len(train_data)

In [ ]:
 len(test_data)

In [ ]:
 train_data[10]

**3.13 Decoding newswires back to text**

In [ ]:
word_index = reuters.get_word_index()
reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])
decoded_newswire = ' '.join([reverse_word_index.get(i - 3, '?') for i in
train_data[0]])

In [ ]:
decoded_newswire

In [ ]:
train_labels[10]

 **3.14 Encoding the data**

You can vectorize the data with the one-hot encoder

In [ ]:
import numpy as np
def vectorize_sequences(sequences, dimension=10000):
  results = np.zeros((len(sequences), dimension))
  for i, sequence in enumerate(sequences):
    results[i, sequence] = 1.
  return results


In [ ]:
x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)

In [ ]:
def to_one_hot(labels, dimension=46):
  results = np.zeros((len(labels), dimension))
  for i, label in enumerate(labels):
    results[i, label] = 1.
  return results


In [ ]:
one_hot_train_labels = to_one_hot(train_labels)
one_hot_test_labels = to_one_hot(test_labels)

Note that there is a built-in way to do this in Keras

In [ ]:
from keras.utils import to_categorical

one_hot_train_labels = to_categorical(train_labels)
one_hot_test_labels = to_categorical(test_labels)

In [ ]:
one_hot_train_labels[0]

**3.15 Model definition**

In [ ]:
from keras import losses, metrics, optimizers


In [ ]:
model = models.Sequential()
model.add(layers.Dense(64, activation='relu', input_shape=(10000,)))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(46, activation='softmax'))

**3.16 Compiling the model**

In [ ]:
model.compile(
    optimizer=optimizers.RMSprop(
        learning_rate=0.001,
    ),
    loss=losses.BinaryCrossentropy(),
    metrics=[
        metrics.BinaryAccuracy(
            name="accuracy",
        )
    ],
)

**3.5.4 Validating your approach**

Let’s set apart 1,000 samples in the training data to use as a validation set.

 **3.17 Setting aside a validation set**

In [ ]:
x_val = x_train[:1000]
partial_x_train = x_train[1000:]
y_val = one_hot_train_labels[:1000]
partial_y_train = one_hot_train_labels[1000:]

**3.18 Training the model**

In [ ]:
history = model.fit(partial_x_train, partial_y_train, epochs=20, batch_size=512, validation_data=(x_val, y_val))

**3.19 Plotting the training and validation loss**

In [ ]:
import matplotlib.pyplot as plt
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'bo', label='Training loss')
plt.plot(epochs, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

**3.20 Plotting the training and validation accuracy**

In [ ]:
plt.clf()
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
plt.plot(epochs, acc, 'bo', label='Training acc')
plt.plot(epochs, val_acc, 'b', label='Validation acc')
plt.title('Training and validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

**PS:**The network begins to overfit after nine epochs. Let’s train a new network from scratch for nine epochs and then evaluate it on the test set

**3.21 Retraining a model from scratch**

In [ ]:
model = models.Sequential()
model.add(layers.Dense(64, activation='relu', input_shape=(10000,)))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(46, activation='softmax'))
model.compile(optimizer='rmsprop', loss='categorical_crossentropy',metrics=['accuracy'])
model.fit(partial_x_train, partial_y_train,epochs=9,batch_size=512,validation_data=(x_val, y_val))
results = model.evaluate(x_test, one_hot_test_labels)

In [ ]:
results

**3.22 Generating predictions for new data**

In [ ]:
predictions = model.predict(x_test)

Each entry in predictions is a vector of length 46:

In [ ]:
predictions[0].shape

The coefficients in this vector sum to 1:

In [ ]:
np.sum(predictions[0])

The largest entry is the predicted class—the class with the highest probability:

In [ ]:
np.argmax(predictions[0])

# **3.6 Predicting house prices: a regression example**

**3.24 Loading the Boston housing dataset**

In [ ]:
from keras.datasets import boston_housing
(train_data, train_targets), (test_data, test_targets) = boston_housing.load_data()

In [ ]:
train_data.shape

In [ ]:
test_data.shape

In [ ]:
train_data[:5]

In [ ]:
train_targets

**3.25 Normalizing the data**

In [ ]:
mean = train_data.mean(axis=0)
train_data -= mean
std = train_data.std(axis=0)
train_data /= std
test_data -= mean
test_data /= std

In [ ]:
train_data[:5]

# **3.6.3 Building your network**

**3.26 Model definition**

In [ ]:
from keras import models
from keras import layers
def build_model():
  model = models.Sequential()
  model.add(layers.Dense(64, activation='relu',   input_shape=(train_data.shape[1],)))
  model.add(layers.Dense(64, activation='relu'))
  model.add(layers.Dense(1))
  model.compile(optimizer='rmsprop', loss='mse', metrics=['mae'])
  return model

In [ ]:
train_data[1].shape

In [ ]:
train_data.shape[1]

**3.27 K-fold validation**

In [ ]:
import numpy as np
k=4
num_val_samples = len(train_data) // k
num_epochs = 100
all_scores = []
for i in range(k):
  print('processing fold #', i)
  val_data = train_data[i * num_val_samples: (i + 1) * num_val_samples]
  val_targets = train_targets[i * num_val_samples: (i + 1) * num_val_samples]
  partial_train_data = np.concatenate(  [train_data[:i * num_val_samples],  train_data[(i + 1) * num_val_samples:]],  axis=0)
  partial_train_targets = np.concatenate(  [train_targets[:i * num_val_samples],  train_targets[(i + 1) * num_val_samples:]],  axis=0)
  model = build_model()
  model.fit(partial_train_data, partial_train_targets,   epochs=num_epochs, batch_size=1, verbose=0)
  val_mse, val_mae = model.evaluate(val_data, val_targets, verbose=0)
  all_scores.append(val_mae)

In [ ]:
all_scores

In [ ]:
np.mean(all_scores)